# Data Preprocessing — Swindon LSOA GVA Project

**Topic:** *From Enterprise Structure to Economic Output: How SME Composition, Economic Resilience, and Employment Shape Local GVA in Swindon*

This notebook prepares the raw dataset for analysis. It reads `features.csv`, restricts the sample to **Swindon LSOAs**, selects the model variables, handles missing values, and creates the outcome variable `log_GVA`.

**Kernel (Cursor):** Select **DS final project (.venv)** — Python 3.11, project-only env (fastest to connect in Cursor).

---

## Input / Output

| File | Description |
|------|-------------|
| `features.csv` | Raw data (unchanged) |
| `swindon_model_ready.csv` | Cleaned modelling dataset (output) |

---

## Preprocessing steps

1. Load `features.csv`
2. Filter to Swindon LSOAs only
3. Remove duplicate `LSOA21CD` records
4. Select model variables (Y, X, mediators, controls)
5. Drop rows with missing GVA
6. Create `log_GVA = log(LSOA GVA Estimates (millions))`
7. Encode `Urban_rura` as a binary variable
8. Run sanity checks (missing values, outliers, MSOA consistency)
9. Save cleaned dataset

---

## Model variables

| Role | Variable |
|------|----------|
| **Y** | `log(LSOA GVA Estimates (millions))` |
| **X1** | `LU_pct_large_2025_msoa` |
| **X2** | `LU_pct_micro_2025_msoa` |
| **X3** | `share_enterprises_kibs_2025_msoa` |
| **M1** | `turnover_diversity_1-HHI_2025_msoa` |
| **M2** | `employment_rate_per_pop` |
| **C1** | `Urban_rura` |
| **C2** | `Pct_Working_Age` |

---

## ID columns (not used as regression predictors)

Analysis unit is **LSOA** (one row = one LSOA). ID columns are kept for labelling and later analysis steps only.

| Column | Level | Why keep it |
|--------|-------|-------------|
| `LSOA21CD` | LSOA | Unique neighbourhood code; used for dedup and record matching |
| `LSOA21NM` | LSOA | Readable neighbourhood name; useful for tables, maps, and case studies |
| `MSOA21CD` | MSOA | Parent area code; **used later for cluster-robust standard errors** because business-structure variables (X1–X3, M1) are measured at MSOA level and repeat across LSOAs within the same MSOA |
| `MSOA21NM` | MSOA | Readable parent area name; **optional**, mainly for reporting |

---

## Notes

- MSOA-level business variables are already merged onto LSOA rows in `features.csv`.
- Do **not** include all four `LU_pct_*` size shares in the same regression (they sum to 1).
- GVA is from **2023**; business variables are from **2025**; employment is from **2024** — document this as a limitation in the final report.

In [1]:
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
print("Python:", sys.executable)
print("Cell 1 OK — imports successful")

Python: /Users/bryanlu/Desktop/DS final project/.venv/bin/python
Cell 1 OK — imports successful


In [2]:
df = pd.read_csv("features.csv")
print("Shape:", df.shape)
df.head()

Shape: (1129, 83)


,LSOA21CD,LSOA21NM,LSOA21NMW,RUC21CD,RUC21NM,Urban_rura,Shape_Leng,Shape__Are,Shape__Len,MSOA21CD,...,share_enterprises_industrial_2025_msoa,share_enterprises_kibs_2025_msoa,share_enterprises_anchor_2025_msoa,enterprises_per_1k_residents_2025,share_enterprises_public_2025_msoa,share_enterprises_private_2025_msoa,bus_stop_count,area_km2,bus_stop_density,dist_to_primary_m
0,E01015471,Swindon 028A,NaN,UN1,Urban: Nearer to a major town or city,Urban,4013.589687,5.013996e+05,4013.589687,E02006849,...,0.020833,0.458333,0.083333,105.448155,0.000000,1.000000,5,1.298791,3.849735,617.714131
1,E01015473,Swindon 008A,NaN,RSN1,Smaller rural: Nearer to a major town or city,Rural,25688.031740,1.524397e+07,25688.031738,E02003219,...,0.045752,0.405229,0.065359,351.401011,0.006536,0.993464,37,39.476058,0.937277,574.697536
2,E01015475,Swindon 015A,NaN,UN1,Urban: Nearer to a major town or city,Urban,2099.250451,2.145225e+05,2099.250451,E02003226,...,0.016129,0.290323,0.064516,124.298316,0.000000,1.000000,10,0.554698,18.027840,350.510113
3,E01015477,Swindon 015C,NaN,UN1,Urban: Nearer to a major town or city,Urban,3276.800336,2.405209e+05,3276.800336,E02003226,...,0.016129,0.290323,0.064516,171.365395,0.000000,1.000000,5,0.621788,8.041327,473.126804
4,E01015478,Swindon 021A,NaN,UN1,Urban: Nearer to a major town or city,Urban,4188.002606,6.194380e+05,4188.002606,E02003232,...,0.033898,0.406780,0.067797,181.315304,0.000000,1.000000,6,1.601008,3.747640,58.398219


In [3]:
df = df[df["LSOA21NM"].str.contains("Swindon", na=False)].copy()
print("Shape after Swindon filter:", df.shape)
df["LSOA21NM"].head()

Shape after Swindon filter: (140, 83)


0    Swindon 028A
1    Swindon 008A
2    Swindon 015A
3    Swindon 015C
4    Swindon 021A
Name: LSOA21NM, dtype: str

In [5]:
df = df.drop_duplicates(subset="LSOA21CD", keep="first")
print("Shape after dedup:", df.shape)

Shape after dedup: (140, 83)


In [7]:
cols = [
    # IDs — LSOA level (analysis unit)
    "LSOA21CD",   # unique LSOA code
    "LSOA21NM",   # readable LSOA name (tables, maps, case studies)
    # IDs — MSOA level (parent area; not the analysis unit)
    "MSOA21CD",   # used later for cluster-robust standard errors (MSOA-level X variables repeat within MSOA)
    "MSOA21NM",   # optional; readable MSOA name for reporting
    # Y
    "LSOA GVA Estimates (millions)",
    # X & mediators
    "LU_pct_large_2025_msoa",
    "LU_pct_micro_2025_msoa",
    "share_enterprises_kibs_2025_msoa",
    "turnover_diversity_1-HHI_2025_msoa",
    "employment_rate_per_pop",
    # Controls
    "Urban_rura",
    "Pct_Working_Age",
]

df = df[cols]
print("Shape after selecting columns:", df.shape)
df.head()

Shape after selecting columns: (140, 12)


,LSOA21CD,LSOA21NM,MSOA21CD,MSOA21NM,LSOA GVA Estimates (millions),LU_pct_large_2025_msoa,LU_pct_micro_2025_msoa,share_enterprises_kibs_2025_msoa,turnover_diversity_1-HHI_2025_msoa,employment_rate_per_pop,Urban_rura,Pct_Working_Age
0,E01015471,Swindon 028A,E02006849,Swindon 028,15.064,0.000000,0.844828,0.458333,0.770833,0.035149,Urban,0.667399
1,E01015473,Swindon 008A,E02003219,Swindon 008,495.260,0.011236,0.820225,0.405229,0.816569,4.708314,Rural,0.612770
2,E01015475,Swindon 015A,E02003226,Swindon 015,99.694,0.011765,0.823529,0.290323,0.830385,0.701684,Urban,0.712911
3,E01015477,Swindon 015C,E02003226,Swindon 015,19.806,0.011765,0.823529,0.290323,0.830385,0.138198,Urban,0.711996
4,E01015478,Swindon 021A,E02003232,Swindon 021,194.209,0.000000,0.895522,0.406780,0.765755,0.276583,Urban,0.695759


In [11]:
GVA_COL = "LSOA GVA Estimates (millions)"

before = len(df)
df = df.dropna(subset=[GVA_COL])
print(f"Dropped {before - len(df)} rows → shape: {df.shape}")

df["log_GVA"] = np.log(df[GVA_COL])
print("After log_GVA:", df.shape)
df[[GVA_COL, "log_GVA"]].head()

Dropped 0 rows → shape: (127, 12)
After log_GVA: (127, 13)


,LSOA GVA Estimates (millions),log_GVA
0,15.064,2.712308
1,495.260,6.205083
2,99.694,4.602105
3,19.806,2.985985
4,194.209,5.268935


In [14]:
df["Urban_rura_binary"] = (df["Urban_rura"] == "Urban").astype(int)

print(df["Urban_rura"].value_counts())
print(df["Urban_rura_binary"].value_counts())
print("Shape:", df.shape)

df.isnull().sum()

Urban_rura
Urban    113
Rural     14
Name: count, dtype: int64
Urban_rura_binary
1    113
0     14
Name: count, dtype: int64
Shape: (127, 14)


LSOA21CD                              0
LSOA21NM                              0
MSOA21CD                              0
MSOA21NM                              0
LSOA GVA Estimates (millions)         0
LU_pct_large_2025_msoa                0
LU_pct_micro_2025_msoa                0
share_enterprises_kibs_2025_msoa      0
turnover_diversity_1-HHI_2025_msoa    0
employment_rate_per_pop               0
Urban_rura                            0
Pct_Working_Age                       0
log_GVA                               0
Urban_rura_binary                     0
dtype: int64

In [15]:
df.to_csv("swindon_model_ready.csv", index=False)